## Simple Loan Officier Agent

This lab will develop a simple LLM agent that can take in an inital prompt, which tells it who it is, and a second prompt telling it how to handle user input. The scenario for this notebook is a loan officier that receives applicant information, but the prompts can be swapped out for any purpose.

In [1]:
### Environtment setup
## Necessary for Colab, not necessary for course environment
## % pip install -q langchain langchain-nvidia-ai-endpoints gradio

import os
os.environ["NVIDIA_API_KEY"] = "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"

## If you encounter a typing-extensions issue, restart your runtime and try again
from langchain_nvidia_ai_endpoints import ChatNVIDIA
# ChatNVIDIA.get_available_models()

### Imports and Preprocessing

These imports make it possible to build the agent.  

It is using NVIDIA's LangChain endpoints (`langchain_nvidia_ai_endpoints`) and accesses the LLM box to access hosted models with `ChatNVIDIA`.

In [2]:
from langchain_nvidia_ai_endpoints import ChatNVIDIA

Since the LLM returns a message and not a string, `StrOutputParser` filters only the text.

In [3]:
from langchain_core.output_parsers import StrOutputParser

`ChatPromptTemplate` allows us to write prompts with blanks like `{input}` and `{audience}`. Without this, we would have to manually build the prompt string every time.

It provides a filled-in message ready for the LLM.

In [4]:
from langchain_core.prompts import ChatPromptTemplate

### LLM Selection, Prompts and Chaining

In [5]:
from dotenv import load_dotenv

load_dotenv()
print("OLLAMA_MODEL:", os.getenv("OLLAMA_MODEL", "qwen2.5:7b"))

OLLAMA_MODEL: qwen2.5:7b


Next we'll need to choose the LLM to work with, set the prompts and chain it all together.

In [6]:
# Choosing the LLM to work with
# llm = ChatNVIDIA(model="nvidia/llama-3.1-nemotron-nano-8b-v1")

llm = ChatOllama(
    model=os.getenv("OLLAMA_MODEL", "qwen2.5:7b"),
    base_url=os.getenv("OLLAMA_BASE_URL", "http://192.168.1.193:11434"),
)

This first prompt tells to the LLM <i>who it is</i> and <i>what to do</i>.

`.from_messages` takes the list of labeled messages and turns them into a proper template the chain can use.

In [7]:
summary_prompt = ChatPromptTemplate.from_messages([
    # "system" is the background instructions
    #    that sets the scene
    ("system", "You are a senior loan officer. Analyze raw loan application data and produce a structured summary. Extract the applicant name, annual income, debt-to-income ratio, credit score, and loan amount, risk level and a one-line recommendation. Use bullet points."),
    # "user" is the human speaking
    ("user", "APPLICATION DATA:\n{input}")
])

The second prompt specifies what to preserve and what to change. It's purpose is to prevent the LLM from inventing numbers, loosely applying risk levels or presenting different figures for different clients. This prompt keeps the agent honest across all that it rewrites.

In [8]:
rewrite_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a financial communications specialist. "
               "Rewrite the internal loan summary for the specified audience. "
               "Keep all facts — do not add or remove financial details. "
               "Adjust only the tone, language, and emphasis to suit the audience."),
    ("user", "ORIGINAL SUMMARY:\n{input}"
                "\n\nTARGET AUDIENCE:\n{audience}")
])

So bascially, `prompt1` tells the LLm what to do, and `prompt2` tells the LLM what it cannot do.

Now that the two prompts are set, they can be chained together along with `StrOutputParser()`.

In [9]:
summary_chain = summary_prompt | llm | StrOutputParser()
rewrite_chain = rewrite_prompt | llm | StrOutputParser()

### Generator Function

Instead of returning one value and stopping, a <b>generator function</b> `yield`s multiple values over time - in this case, one token at a time as the LLM produces them.

The purpose of this generator function is to manage the conversation between the user and the two chains. It looks at `history` to figure out where in the conversation it is, picks the right chain and streams the response back token by token.

If no summary exists in the history yet, it calls `summary_chain`. If a summary does already exist, it calls `rewrite_chain`.

In [10]:
def routed_chat_stream(message, history, return_buffer=True):
    # "message" contains whatever the user typed in the input box
    # "history" is the full conversation so far, as a list of dictionaries
    # "return_buffer" controls whether athe function yeilds the whole buffer
    #    or just each individual token as the LLM genertaes it

    first_summary = None # no summary on first turn
    ## extracted from history if a summary already exists

    for entry in history: # for each entry in the history
            if entry.get("role") == "assistant":
                # if the role is identified as "assistant" (the agent)
                content = entry.get("content", "")
                # then pull from the content part of the dict,
                #    the raw data payload ("content") and if
                #    there is no content, return empty string ("")
                if "Let me think about this." in content:
                    # if this phrase is found in the raw data payload
                    parts = content.split("\n\n")
                    #    then split by the "think" preface and the rewrite
                    if len(parts) > 2: # if all three sections exist
                        first_summary = "\n\n".join(parts[1:-1]) 
                        # grab the middle section by finding the first
                        #    and last, then joining the middle
                        break

    if first_summary is None: # if it is the first message to "user"
        buffer = "I'll analyze the applicant data and provide you a summary. Let me think about this.\n\n"
        # show the buffer message which includes the trigger phrase
        yield buffer if return_buffer else buffer
        # send the buffer to whoever is listening
        #    whether it is gradio or a terminal

        inst_out = ""
        chat_gen = summary_chain.stream({"input" : message})
        # .stream() is telling LangChain to ouput messages
        #   by token instead of by message
        for token in chat_gen:
            inst_out += token # accumulate raw LLM output
            buffer += token # accumulate response the user sees
            yield buffer if return_buffer else token

        passage = "\n\nThere. A finished summary. I can tailor this to a specific audience. Who is it for?"
        buffer += passage # add the passage to the buffer
        yield buffer if return_buffer else passage

    else:

        # Invoke the second chain to generate the rewrite
        inst_out = ""
        buffer = f"Sure! Here you go!\n\n"
        yield buffer

        chat_gen = rewrite_chain.stream({"audience": message, "input": first_summary})

        # Iterate over stream generator
        for token in chat_gen:
            inst_out += token
            buffer += token
            yield buffer if return_buffer else token

        passage = "\n\nDone. Let me know any more audiences."
        buffer += passage
        yield buffer if return_buffer else passage

### Terminal Wrapper

Next up is building the terminal interface that the angent will run in. It appends the content to `history` which the previous function sifts through to find entries and build `content`. 

In [11]:
def terminal_chat(chat_stream, history=[]):

    ## print the starter message
    for entry in history:
        # below line pulls the role from the history dictionary
        #    and capitalizes the first letter (i.e, user >> User)
        role = entry.get("role").capitalize()
        # prints the "[ User ]:, [ Assistant ]:" preface
        print(f"\n[ {role} ]:", entry.get("content"))

    # run until user types quit
    while True: # func doesn't know when user will stop
        user_input = input("\n[ You ]: ")

        # if user enters any one of these keywords
        if user_input.lower() in ["quit", "exit", "q"]:
            print("Session ended.") # then print "session ended"
            break # and break out of the loop

        # Add a user message to history
        history.append({"role": "user", "content": user_input})
    
        print("\n [ Agent ]: ")
        full_response = ""
    
        # Pull in tokens from the agent function
        for token in chat_stream(user_input, history, return_buffer=False):
            # ^^ called to "routed_chat_stream" at the bottom
            print(token, end='')
            full_response += token
    
        # Add agent response to history
        history.append({"role": "assistant", "content": full_response})
        print("\n")

# Starter message
history = [{
    "role": "assistant",
    "content": "Hi. Please paste in raw application data and I'll reproduce a structured summary. Then tell me which audience to rewrite it for."
}]

# Run the chat
terminal_chat(
    chat_stream=routed_chat_stream,
    history=history
)


[ Assistant ]: Hi. Please paste in raw application data and I'll reproduce a structured summary. Then tell me which audience to rewrite it for.



[ You ]:  John Martinez DOB: 1979-07-22 Job: Software Engineer at TechCorp, 8 years Salary: $115,000/year Monthly expenses: rent $0 (selling current home), car payment $650,  student loans $400, credit cards $280 Credit score: 748 Wants to borrow: $420,000 for a house worth $525,000 Savings/assets: $87,000 checking, 401k ~$210,000 No bankruptcies, no late payments in 5 years



 [ Agent ]: 
I'll analyze the applicant data and provide you a summary. Let me think about this.

- **Applicant Name:** John Martinez
- **Annual Income:** $115,000/year
- **Debt-to-Income Ratio:** 
  - Monthly Debt: $1330 (car payment + student loans + credit cards)
  - Gross Monthly Income: $9583.33 (annual income / 12 months)
  - DTI: 14% ($1330 / $9583.33)
- **Credit Score:** 748
- **Loan Amount Requested:** $420,000
- **House Value:** $525,000
- **Savings/Assets:** 
  - Checking: $87,000
  - 401k: ~$210,000
- **Risk Level:** Low to Moderate
- **Recommendation:** Approve loan; good credit score, low DTI, stable employment, and sufficient savings.

There. A finished summary. I can tailor this to a specific audience. Who is it for?




[ You ]:  A vc investor



 [ Agent ]: 
Sure! Here you go!

**Loan Summary for John Martinez**

*Investment Opportunity:*

We are pleased to present an investment opportunity in the form of a secured loan application from Mr. John Martinez. Here is a detailed overview:

- **Annual Income:** $115,000/year
- **Monthly Debt Payments:**
  - Car Payment: $400
  - Student Loans: $650
  - Credit Cards: $280
  - Total Monthly Debt: $1330

- **Income Analysis:**
  - Gross Monthly Income: $9,583.33 (annual income divided by 12 months)
  - Debt-to-Income Ratio: 14% ($1330 / $9583.33)

- **Creditworthiness:** 
  - Credit Score: 748

- **Requested Loan Amount:** $420,000
- **Property Value:** House valued at $525,000

- **Financial Reserves:**
  - Checking Account: $87,000
  - 401k Savings: ~$210,000

- **Risk Assessment:** 
  - Risk Level: Low to Moderate

*Recommendation:*

Given Mr. Martinez’s strong financial profile—including a solid credit score, manageable debt-to-income ratio, stable income, and substantial savings—


[ You ]:  An accountant.



 [ Agent ]: 
Sure! Here you go!

**Internal Loan Summary for John Martinez**

---

*Applicant Name:* John Martinez

*Annual Income:* $115,000/year

*Debt-to-Income (DTI) Ratio:*
- Monthly Debt: $1330 (comprising car payment, student loans, and credit card obligations)
- Gross Monthly Income: $9583.33 ($115,000/12 months)
- DTI: 14% ($1330/$9583.33)

*Credit Score:* 748

*Loan Amount Requested:* $420,000

*House Value:* $525,000

*Savings/Assets:*
- Checking Account: $87,000
- 401k Plan: Approximately $210,000

*Risk Level:* Low to Moderate

*Recommendation:* Approve loan. Given John's strong credit score of 748, his low DTI ratio at 14%, stable employment income, and substantial savings in both a checking account and his 401k, the risk level is appropriately categorized as low to moderate.

---

Feel free to review these details and make any necessary adjustments before proceeding with the approval process.

Done. Let me know any more audiences.




[ You ]:  q


Session ended.


from langchain_nvidia_ai_endpoints import ChatNVIDIA
models = ChatNVIDIA.get_available_models()
for m in models:
    print(m.id)